In [1]:
# 패키지 설치
!pip install bertopic sentence-transformers kiwipiepy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 10.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 19.4 MB/s eta 0:00:00
  Created wheel for kiwipiepy_model: filename=kiwipiepy_model-0.23.0-py3-none-any.whl size=88067826 sha256=1ee2b06bbf489f6f34e24813a13fd5844a9eda3276e44852a7bc521181315898
  Stored in directory: /root/.cache/pip/wheels/f2/94/da/ff88b4c2305cd1f3effc8001b5f42f16dc9931bcd84d1e77c3
Successfully built kiwipiepy_model


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import os

!find /content/drive/MyDrive -name "cnp_processed.csv" 2>/dev/null

Mounted at /content/drive
/content/drive/MyDrive/cnp_processed.csv
^C


In [4]:
df = pd.read_csv("/content/drive/MyDrive/cnp_processed.csv")
print(f"로드 완료: {len(df)}건")
print(df.columns.tolist())

로드 완료: 6522건
['source', 'date', 'query', 'raw_text', 'unigram', 'bigram', 'unibi_mix', 'adj_noun', 'video_title', 'likes']


In [5]:
import ast
import warnings
warnings.filterwarnings("ignore")
from kiwipiepy import Kiwi
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

# ── CNP 필터링 ──
CNP_KEYWORDS = {
    "차앤박", "CNP", "프로폴리스", "앰플", "PDRN",
    "안티포어", "트러블", "카밍", "클렌징", "세럼", "더마"
}
df_cnp = df[df["raw_text"].apply(
    lambda x: any(kw in str(x) for kw in CNP_KEYWORDS)
)].copy().reset_index(drop=True)
print(f"CNP 필터링 후: {len(df_cnp)}건")

# ── 문서 준비 ──
docs = df_cnp["raw_text"].fillna("").tolist()
docs = [d for d in docs if len(d) >= 10]
print(f"유효 문서: {len(docs)}건")

# ── 임베딩 모델 ──
print("\n임베딩 모델 로드 중...")
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# ── 한국어 토크나이저 ──
kiwi = Kiwi()

def korean_tokenizer(text):
    try:
        result = kiwi.analyze(text)
        tokens = [
            token.form for token in result[0][0]
            if token.tag in ("NNG", "NNP") and len(token.form) >= 2
        ]
        return tokens if tokens else text.split()
    except:
        return text.split()

vectorizer = CountVectorizer(
    tokenizer=korean_tokenizer,
    min_df=3,
    max_df=0.85,
    max_features=5000
)

# ── BERTopic 학습 ──
print("BERTopic 학습 중...")
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer,
    representation_model=KeyBERTInspired(),
    nr_topics="auto",
    min_topic_size=15,
    verbose=True
)

topics, probs = topic_model.fit_transform(docs)
print(f"\n발견된 토픽 수: {len(set(topics)) - 1}")

CNP 필터링 후: 3013건
유효 문서: 3010건

임베딩 모델 로드 중...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

2026-04-14 06:11:07,702 - BERTopic - Embedding - Transforming documents to embeddings.


BERTopic 학습 중...


Batches:   0%|          | 0/95 [00:00<?, ?it/s]

2026-04-14 06:11:13,522 - BERTopic - Embedding - Completed ✓
2026-04-14 06:11:13,523 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-14 06:11:38,386 - BERTopic - Dimensionality - Completed ✓
2026-04-14 06:11:38,387 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-14 06:11:38,489 - BERTopic - Cluster - Completed ✓
2026-04-14 06:11:38,490 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-04-14 06:11:49,905 - BERTopic - Representation - Completed ✓
2026-04-14 06:11:49,906 - BERTopic - Topic reduction - Reducing number of topics
2026-04-14 06:11:49,917 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-14 06:12:04,729 - BERTopic - Representation - Completed ✓
2026-04-14 06:12:04,734 - BERTopic - Topic reduction - Reduced number of topics from 29 to 15



발견된 토픽 수: 14


In [6]:
# ── 토픽 요약 ──
topic_info = topic_model.get_topic_info()
print("=== BERTopic 토픽 요약 ===")
print(topic_info[topic_info["Topic"] != -1][["Topic", "Count", "Name"]].to_string(index=False))

print("\n=== 토픽별 키워드 ===")
for topic_id in sorted(set(topics)):
    if topic_id == -1:
        continue
    keywords = topic_model.get_topic(topic_id)
    kw_str = ", ".join([kw for kw, _ in keywords[:8]])
    count = topics.count(topic_id)
    print(f"Topic {topic_id} ({count}건): {kw_str}")

=== BERTopic 토픽 요약 ===
 Topic  Count                   Name
     0   1113      0_피부과_피부_화장품_스킨케어
     1    284      1_콜라겐_피부_화장품_스킨케어
     2    185         2_가격_구매_구매자_반값
     3    102         3_샘플_화율_리들샷_캡슐
     4     75         4_블랙_상자_패키지_키트
     5     56   5_바이오더마_콜라겐_유산균_시카페어
     6     47         6_홈케어_착용_케어_후기
     7     36      7_화장품_피부과_피부염_추출물
     8     34      8_썬크림_자외선_선크림_레이저
     9     32     9_화장품_뷰티_뷰티템_클래리파잉
    10     27 10_헤라샘플_클리닉_리플렉팅_링클코렉터
    11     20       11_콜라_구매_스킨푸드_쫀쫀
    12     18     12_매장_스프레이_코스트코_레몬
    13     17      13_스킨케어_피부과_피부_건강

=== 토픽별 키워드 ===
Topic 0 (1113건): 피부과, 피부, 화장품, 스킨케어, 여드름, 마스크, 얼굴, 쫀쫀
Topic 1 (284건): 콜라겐, 피부, 화장품, 스킨케어, 리뷰, 레티놀, 판테놀, 개선
Topic 2 (185건): 가격, 구매, 구매자, 반값, 쿠폰, 마켓, 최대, 상품
Topic 3 (102건): 샘플, 화율, 리들샷, 캡슐, 항목, 게시, 패드, 기초
Topic 4 (75건): 블랙, 상자, 패키지, 키트, 노폐물, 면봉, 구성, 세트
Topic 5 (56건): 바이오더마, 콜라겐, 유산균, 시카페어, 이마트, 수분, 워터, 효소
Topic 6 (47건): 홈케어, 착용, 케어, 후기, 전문, 포스팅, 효과, 르세라핌
Topic 7 (36건): 화장품, 피부과, 피부염, 추출물, 피부, 약국, 더마코스메틱,

In [7]:
import json

# ── LDA × BERTopic 교차검증 ──
LDA_TOPICS = {
    "블랙헤드_안티포어": ["블랙", "헤드", "안티포어", "클리어", "키트"],
    "프로폴리스_앰플": ["프로폴리스", "앰플", "미스트", "에너지"],
    "트러블_이탈": ["트러블", "여드름", "자극", "카밍"],
    "더마코스메틱": ["더마", "피부과", "성분", "추출물"],
}

print("=== LDA × BERTopic 교차검증 ===\n")
consensus = []
for lda_name, lda_kws in LDA_TOPICS.items():
    print(f"[ LDA: {lda_name} ]")
    found = False
    for topic_id in sorted(set(topics)):
        if topic_id == -1:
            continue
        bert_kws = [kw for kw, _ in topic_model.get_topic(topic_id)[:10]]
        overlap = set(lda_kws) & set(bert_kws)
        if overlap:
            count = topics.count(topic_id)
            print(f"  → BERTopic Topic {topic_id} ({count}건) 합의: {overlap}")
            consensus.append({
                "lda_topic": lda_name,
                "bertopic_id": topic_id,
                "overlap_keywords": list(overlap),
                "bertopic_count": count,
                "confidence": "high"
            })
            found = True
    if not found:
        print(f"  → BERTopic 합의 없음 (LDA 단독 신호)")
        consensus.append({
            "lda_topic": lda_name,
            "bertopic_id": None,
            "overlap_keywords": [],
            "bertopic_count": 0,
            "confidence": "low"
        })
    print()

# ── 저장 ──
topic_info_filtered = topic_info[topic_info["Topic"] != -1]
topic_info_filtered.to_csv("cnp_bertopic_results.csv",
                            index=False, encoding="utf-8-sig")

df_cnp_result = df_cnp.iloc[:len(topics)].copy()
df_cnp_result["bertopic_id"] = topics
df_cnp_result.to_csv("cnp_bertopic_documents.csv",
                      index=False, encoding="utf-8-sig")

topic_keywords = {}
for tid in sorted(set(topics)):
    if tid == -1:
        continue
    topic_keywords[str(tid)] = [kw for kw, _ in topic_model.get_topic(tid)[:10]]

with open("cnp_bertopic_keywords.json", "w", encoding="utf-8") as f:
    json.dump(topic_keywords, f, ensure_ascii=False, indent=2)

consensus_df = pd.DataFrame(consensus)
consensus_df.to_csv("cnp_lda_bertopic_consensus.csv",
                     index=False, encoding="utf-8-sig")

print("저장 완료:")
print("  cnp_bertopic_results.csv")
print("  cnp_bertopic_documents.csv")
print("  cnp_bertopic_keywords.json")
print("  cnp_lda_bertopic_consensus.csv")

# ── 다운로드 ──
from google.colab import files
files.download("cnp_bertopic_results.csv")
files.download("cnp_bertopic_documents.csv")
files.download("cnp_bertopic_keywords.json")
files.download("cnp_lda_bertopic_consensus.csv")

=== LDA × BERTopic 교차검증 ===

[ LDA: 블랙헤드_안티포어 ]
  → BERTopic Topic 4 (75건) 합의: {'블랙', '키트'}
  → BERTopic Topic 11 (20건) 합의: {'클리어'}

[ LDA: 프로폴리스_앰플 ]
  → BERTopic 합의 없음 (LDA 단독 신호)

[ LDA: 트러블_이탈 ]
  → BERTopic Topic 0 (1113건) 합의: {'여드름'}

[ LDA: 더마코스메틱 ]
  → BERTopic Topic 0 (1113건) 합의: {'피부과'}
  → BERTopic Topic 7 (36건) 합의: {'피부과', '추출물'}
  → BERTopic Topic 13 (17건) 합의: {'피부과'}

저장 완료:
  cnp_bertopic_results.csv
  cnp_bertopic_documents.csv
  cnp_bertopic_keywords.json
  cnp_lda_bertopic_consensus.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>